# 정적 웹페이지 페이지네이션 수집

## 수집 배치별 HTML 폴더 관리

Books to Scrape 웹사이트의 여러 페이지에 순차적으로 HTTP 요청을 보내고,
서버가 반환한 원본 HTML을 **수집 실행 단위의 배치 폴더**에 저장한다.

### 변경 사항

```text
Step 01
data/raw/html/
├─ books_page_001_20260809_224616.html
├─ books_page_002_20260809_224616.html
└─ books_page_003_20260809_224616.html

        ⬇️

Step 02
data/raw/html/
└─ 20260809_224616/
   ├─ books_page_001.html
   ├─ books_page_002.html
   └─ books_page_003.html
```

같은 크롤링 실행에서 생성된 HTML 파일을 하나의 폴더로 묶어
수집 배치를 쉽게 구분하고 관리할 수 있다.

- https://books.toscrape.com/


In [1]:
"""
Books to Scrape 웹사이트의 정적 페이지를
페이지네이션 방식으로 수집하는 모듈입니다.

수집 시작 시각을 이름으로 사용하는 배치 폴더를 생성하고,
같은 실행에서 수집한 HTML 파일을 해당 폴더에 함께 저장합니다.

저장 구조:
    data/raw/html/YYYYMMDD_HHMMSS/
        books_page_001.html
        books_page_002.html
        ...
"""

from datetime import datetime
from pathlib import Path
import time

import requests


In [2]:
## ===========================================================
## 1. 수집 설정
## ===========================================================

## 페이지 URL의 공통 부분
## 예: https://books.toscrape.com/catalogue/page-1.html
BASE_URL = 'https://books.toscrape.com/catalogue/'

## 수집할 페이지 범위
START_PAGE = 1
END_PAGE = 3

## 요청 제한 시간
CONNECT_TIMEOUT = 5
READ_TIMEOUT = 30

## 연속 요청 사이의 대기 시간(초)
REQUEST_INTERVAL = 0.5

## HTTP 요청 헤더
HEADERS = {'User-Agent': 'EducationalDataCollector/1.0'}


In [3]:
## ===========================================================
## 2. 기본 저장 경로 설정
## ===========================================================

PROJECT_DIR = Path.cwd().resolve().parents[1]

## 모든 HTML 수집 배치가 저장되는 기본 폴더
RAW_HTML_DIR = PROJECT_DIR / 'data' / 'raw' / 'html'


In [4]:
## ===========================================================
## 3. 수집 배치 폴더 생성
## ===========================================================

## 전체 수집 작업의 시작 시각
collected_at = datetime.now()

## 수집 시작 시각을 배치 이름으로 사용
batch_name = collected_at.strftime('%Y%m%d_%H%M%S')

## 같은 수집 작업에서 생성된 HTML 파일을 저장할 배치 폴더
batch_dir = RAW_HTML_DIR / batch_name

batch_dir.mkdir(
    parents=True,   ## 부모 폴더가 없으면 함께 생성
    exist_ok=True,  ## 폴더가 이미 존재해도 오류를 발생시키지 않음
)

## 저장된 HTML 파일 경로 목록
raw_files: list[Path] = []


In [5]:
## ===========================================================
## 4. 페이지 범위 요청 및 HTML 저장
## ===========================================================

try:
    for page in range(START_PAGE, END_PAGE + 1):
        target_url = f'{BASE_URL}page-{page}.html'

        print('=' * 60)
        print(f'{page}페이지 요청 시작')
        print(f'요청 URL : {target_url}')

        response = requests.get(
            target_url,
            headers=HEADERS,
            timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
        )

        ## HTTP 상태 코드가 400번대 또는 500번대이면
        ## requests.exceptions.HTTPError 발생
        response.raise_for_status()

        ## 배치 폴더가 수집 시각을 관리하므로
        ## HTML 파일명에는 페이지 번호만 포함
        raw_file = batch_dir / f'books_page_{page:03d}.html'

        ## 서버가 반환한 원본 HTML 바이트 데이터 저장
        raw_file.write_bytes(response.content)

        ## 저장된 파일 경로 추가
        raw_files.append(raw_file)

        print(f'최종 URL : {response.url}')
        print(f'상태 코드 : {response.status_code}')
        print(f"Content-Type : {response.headers.get('Content-Type')}")
        print(f'응답 인코딩 : {response.encoding}')
        print(f'본문 기준 추정 인코딩 : {response.apparent_encoding}')
        print(f'응답 크기 : {len(response.content):,} bytes')
        print(f'수집 시작 시각 : {collected_at:%Y-%m-%d %H:%M:%S}')
        print(f'원본 HTML 저장 경로 : {raw_file}')

        ## 마지막 페이지가 아니면 다음 요청 전 대기
        if page < END_PAGE:
            time.sleep(REQUEST_INTERVAL)

## ===========================================================
## 5. 요청 오류 처리
## ===========================================================
except requests.exceptions.HTTPError as error:
    print()
    print(f'{page}페이지 HTTP 응답 오류가 발생했습니다.')
    print(f'오류 내용 : {error}')

except requests.exceptions.RequestException as error:
    print()
    print(f'{page}페이지 요청 중 오류가 발생했습니다.')
    print(f'오류 내용 : {error}')

## ===========================================================
## 6. 요청 성공 처리
## ===========================================================
else:
    print()
    print('=' * 60)
    print('웹페이지 수집을 완료했습니다.')
    print('=' * 60)

    print(f'수집 페이지 : {START_PAGE}~{END_PAGE}')
    print(f'수집 페이지 수 : {len(raw_files)}')
    print(f'수집 시작 시각 : {collected_at:%Y-%m-%d %H:%M:%S}')
    print(f'수집 배치명 : {batch_name}')
    print(f'HTML 저장 폴더 : {batch_dir}')


1페이지 요청 시작
요청 URL : https://books.toscrape.com/catalogue/page-1.html
최종 URL : https://books.toscrape.com/catalogue/page-1.html
상태 코드 : 200
Content-Type : text/html
응답 인코딩 : ISO-8859-1
본문 기준 추정 인코딩 : utf-8
응답 크기 : 50,469 bytes
수집 시작 시각 : 2026-08-10 07:29:36
원본 HTML 저장 경로 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\raw\html\20260810_072936\books_page_001.html
2페이지 요청 시작
요청 URL : https://books.toscrape.com/catalogue/page-2.html
최종 URL : https://books.toscrape.com/catalogue/page-2.html
상태 코드 : 200
Content-Type : text/html
응답 인코딩 : ISO-8859-1
본문 기준 추정 인코딩 : utf-8
응답 크기 : 50,877 bytes
수집 시작 시각 : 2026-08-10 07:29:36
원본 HTML 저장 경로 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\raw\html\20260810_072936\books_page_002.html
3페이지 요청 시작
요청 URL : https://books.toscrape.com/catalogue/page-3.html
최종 URL : https://books.toscrape.com/catalogue/page-3.html
상태 코드 : 200
Content-Type : text/html
응답 인코딩 : ISO-8859-1
본문 기준 추정 인코딩 : utf-8
응답 크기 : 51,374 bytes
수집 시작 시각 : 2